# KS Test Drift Detection

This notebook detects distribution drift via the Kolmogorov–Smirnov (KS) test by comparing features across time periods. It computes the KS statistic and p-value between the **training and testing splits** (chronological 64% train / 20% test, consistent with the ML pipeline) using actual demand values from `mle.gold.product_city_demand`, logging results to `mle.monitoring.model_drift_ks_results` for monitoring. A consecutive-month comparison is also logged for dashboard continuity.

---

## 1. Load Data from Gold Table

Load the demand data from the Unity Catalog `mle.gold.product_city_demand` table for analysis.


In [0]:
# Load data
from datetime import datetime
from pyspark.sql.functions import col, month, year
import pandas as pd

# Load table
spark_df = spark.table("mle.gold.product_city_demand")
spark_df = spark_df.withColumn('transaction_month', month(col('transaction_date')))
spark_df = spark_df.withColumn('transaction_year', year(col('transaction_date')))
pd_df = spark_df.select('transaction_date', 'demand_quantity', 'product_name', 'destination_city', 'transaction_month', 'transaction_year').toPandas()

pd_df.head()

## 2. KS Test: Compare Demand Distribution between Training and Testing Data

This step splits the demand data chronologically (64% train / 16% val / 20% test, the same split used by the ML pipeline) and computes the KS statistic and p-value on `demand_quantity` between the training and testing periods. A consecutive-month comparison is also retained for continuity. Results are logged for drift monitoring.

In [0]:
# Compute KS Test: Training vs Testing distribution drift
from scipy.stats import ks_2samp
import pyspark.sql.types as T

# --- Chronological split (64% train / 16% val / 20% test) — same approach as ml_pipeline ---
date_col = pd.to_datetime(pd_df['transaction_date'])
dates = sorted(date_col.unique())
n_dates = len(dates)
if n_dates < 5:
    raise ValueError("Not enough distinct transaction dates to build train/test splits.")
n_train = int(n_dates * 0.64)
n_val = int(n_dates * 0.16)
train_end, test_start = dates[n_train - 1], dates[n_train + n_val]

train_demand = pd_df[date_col <= train_end]['demand_quantity'].dropna()
test_demand = pd_df[date_col >= test_start]['demand_quantity'].dropna()

ks_stat, p_value = ks_2samp(train_demand, test_demand)

print(f"Train: {len(train_demand)} rows ({dates[0].date()} to {train_end.date()})")
print(f"Test:  {len(test_demand)} rows ({test_start.date()} to {dates[-1].date()})")
print(f"KS Statistic: {ks_stat:.4f}\np-value: {p_value:.4f}")
print(f"Drift detected: {'YES' if p_value < 0.05 else 'NO'} (alpha = 0.05)\n")
print("Train distribution:\n", train_demand.describe().round(2), "\n")
print("Test distribution:\n", test_demand.describe().round(2))

# --- Also keep the consecutive-month comparison for dashboard continuity ---
pairs = sorted(set(zip(pd_df['transaction_year'], pd_df['transaction_month'])))
(current_year, current_month) = pairs[-1]
(previous_year, previous_month) = pairs[-2]
cur = pd_df[(pd_df['transaction_year']==current_year) & (pd_df['transaction_month']==current_month)]['demand_quantity'].dropna()
prev = pd_df[(pd_df['transaction_year']==previous_year) & (pd_df['transaction_month']==previous_month)]['demand_quantity'].dropna()
m_ks, m_p = ks_2samp(cur, prev)
print(f"\nConsecutive months {previous_year}-{previous_month:02d} vs {current_year}-{current_month:02d}: KS={m_ks:.4f}, p={m_p:.4f}")

results = [
    {
        "eval_date": str(datetime.now().date()),
        "feature": "demand_quantity",
        "reference_month": train_end.month,
        "reference_year": train_end.year,
        "current_month": dates[-1].month,
        "current_year": dates[-1].year,
        "ks_statistic": float(ks_stat),
        "p_value": float(p_value),
        "comparison_type": "train_vs_test",
        "train_rows": int(len(train_demand)),
        "test_rows": int(len(test_demand)),
    },
    {
        "eval_date": str(datetime.now().date()),
        "feature": "demand_quantity",
        "reference_month": previous_month,
        "reference_year": previous_year,
        "current_month": current_month,
        "current_year": current_year,
        "ks_statistic": float(m_ks),
        "p_value": float(m_p),
        "comparison_type": "consecutive_months",
        "train_rows": None,
        "test_rows": None,
    },
]

schema = T.StructType([
    T.StructField("eval_date", T.StringType()),
    T.StructField("feature", T.StringType()),
    T.StructField("reference_month", T.IntegerType()),
    T.StructField("reference_year", T.IntegerType()),
    T.StructField("current_month", T.IntegerType()),
    T.StructField("current_year", T.IntegerType()),
    T.StructField("ks_statistic", T.DoubleType()),
    T.StructField("p_value", T.DoubleType()),
    T.StructField("comparison_type", T.StringType()),
    T.StructField("train_rows", T.IntegerType()),
    T.StructField("test_rows", T.IntegerType()),
])

spark.sql("CREATE SCHEMA IF NOT EXISTS mle.monitoring")
spark_df_out = spark.createDataFrame(results, schema)
spark_df_out.write.format("delta").option("mergeSchema", "true").mode("append").saveAsTable("mle.monitoring.model_drift_ks_results")
print("\nDrift stats saved to mle.monitoring.model_drift_ks_results (comparison_type = train_vs_test / consecutive_months)")